In [1]:
import pandas as pd
import sqlite3
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

-----------------------------------------

# Primer ingesta, inspección de la estructura de las tablas

In [2]:
# Configuración de visualización
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
sns.set_theme(style="whitegrid")

# 1. Definición de rutas a las fuentes SQLite (.db)
DATA_DIR = Path("../data")

databases = {
    "clientes": DATA_DIR / "clientes.db",
    "crean_aho_cte": DATA_DIR / "crean_aho_cte.db",
    "crean_bolsillos": DATA_DIR / "crean_bolsillos.db",
    "crean_fiducuenta": DATA_DIR / "crean_fiducuenta.db",
    "crean_inv_virtual_cdt": DATA_DIR / "crean_inv_virtual_cdt.db",
    "estimador_ing": DATA_DIR / "estimador_ing.db",
    "invesbot": DATA_DIR / "invesbot.db"
}

# 2. Ingesta y almacenamiento en diccionario de DataFrames
dfs = {}

print("=== INGESTO DE FUENTES DE DATOS ===")
for name, db_path in databases.items():
    if not db_path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {db_path.resolve()}")
    
    conn = sqlite3.connect(db_path)
    # Extraemos el nombre de la tabla interna dinámicamente
    table_name_query = "SELECT name FROM sqlite_master WHERE type='table';"
    table_name = pd.read_sql_query(table_name_query, conn).iloc[0, 0]
    
    # Cargamos el DataFrame
    dfs[name] = pd.read_sql_query(f"SELECT * FROM {table_name}", conn)
    conn.close()
    print(f"Fuente '{name}' cargada con éxito. Registros: {len(dfs[name]):,} | Columnas: {list(dfs[name].columns)}")

=== INGESTO DE FUENTES DE DATOS ===
Fuente 'clientes' cargada con éxito. Registros: 860,231 | Columnas: ['numero_id', 'grupo_edad', 'desc_genero', 'desc_segmento', 'desc_tipo_de_vivienda', 'ingresos_mensuales', 'total_egresos_mensuales', 'total_activos', 'total_pasivos', 'total_patrimonio']
Fuente 'crean_aho_cte' cargada con éxito. Registros: 1,000,000 | Columnas: ['fecha', 'numero_id', 'producto', 'saldo']
Fuente 'crean_bolsillos' cargada con éxito. Registros: 1,000,000 | Columnas: ['fecha', 'numero_id', 'producto', 'saldo']
Fuente 'crean_fiducuenta' cargada con éxito. Registros: 1,000,000 | Columnas: ['fecha', 'numero_id', 'producto', 'saldo']
Fuente 'crean_inv_virtual_cdt' cargada con éxito. Registros: 994,177 | Columnas: ['fecha', 'numero_id', 'producto', 'saldo']
Fuente 'estimador_ing' cargada con éxito. Registros: 745,792 | Columnas: ['numero_id', 'producto', 'estimador_ingreso']
Fuente 'invesbot' cargada con éxito. Registros: 1,000,000 | Columnas: ['fecha', 'numero_id', 'product

--------------------------------------------------------------------

# Analisis inicial con shape, dtype y head()

In [3]:
dfs = {}
resumen_estructura = []

for nombre_tabla, ruta_db in databases.items():
    with sqlite3.connect(ruta_db) as conn:
        df = pd.read_sql(f"SELECT * FROM {nombre_tabla}", conn)
    dfs[nombre_tabla] = df

    print(f"\n=== {nombre_tabla} ===")
    print("shape:", df.shape)
    print("dtypes:")
    print(df.dtypes)
    print("head():")
    display(df.head())

    resumen_estructura.append({
        "tabla": nombre_tabla,
        "filas": df.shape[0],
        "columnas": df.shape[1],
    })

df_resumen_estructura = pd.DataFrame(resumen_estructura).sort_values("tabla")
print("\nResumen de estructuras cargadas:")
display(df_resumen_estructura)


=== clientes ===
shape: (860231, 10)
dtypes:
numero_id                    int64
grupo_edad                     str
desc_genero                    str
desc_segmento                  str
desc_tipo_de_vivienda          str
ingresos_mensuales         float64
total_egresos_mensuales    float64
total_activos              float64
total_pasivos              float64
total_patrimonio           float64
dtype: object
head():


,numero_id,grupo_edad,desc_genero,desc_segmento,desc_tipo_de_vivienda,ingresos_mensuales,total_egresos_mensuales,total_activos,total_pasivos,total_patrimonio
0,8805210490649048784,65+,masculino,preferencial,NaN,29239444.00,30000000.00,145047043000.00,6356702000.00,105422323.00
1,-5723572980375902369,50-65,masculino,preferencial,PROPIA,31024544.00,10000000.00,1133345000.00,83072000.00,1050273000.00
2,-8245474570363424359,65+,masculino,preferencial,PROPIA,2834000.00,500000.00,1073787017.00,0.00,343000000.00
3,-7840506796880772723,65+,femenino,preferencial,PROPIA,28035850.00,0.00,175000000.00,0.00,55000000.00
4,5309731180094827430,36-49,femenino,preferencial,FAMILIAR,3846205.00,2500000.00,758000000.00,0.00,758000000.00



=== crean_aho_cte ===
shape: (1000000, 4)
dtypes:
fecha            str
numero_id      int64
producto         str
saldo        float64
dtype: object
head():


,fecha,numero_id,producto,saldo
0,2026-03-01,-1787624227069906332,CUENTA DE AHORRO,171151.39
1,2025-07-01,2081948080502941032,CUENTA DE AHORRO,7674083.03
2,2026-06-01,-4762164790684256469,CUENTA DE AHORRO,22170.28
3,2025-12-01,7597949800216855292,CUENTA DE AHORRO,1121.97
4,2026-05-01,-5893334273208187262,CUENTA DE AHORRO,10945.58



=== crean_bolsillos ===
shape: (1000000, 4)
dtypes:
fecha            str
numero_id      int64
producto         str
saldo        float64
dtype: object
head():


...

,numero_id,producto,estimador_ingreso
0,8482174354453385212,ESTIMADOR INGRESO,4853570.31
1,8831305677363912137,ESTIMADOR INGRESO,1749148.95
2,8289414305330255763,ESTIMADOR INGRESO,1542841.12
3,5066284055348912520,ESTIMADOR INGRESO,2889280.32
4,4208199011104756958,ESTIMADOR INGRESO,1742505.76



=== invesbot ===
shape: (1000000, 4)
dtypes:
fecha            str
numero_id      int64
producto         str
saldo        float64
dtype: object
head():


,fecha,numero_id,producto,saldo
0,2026-05-11,-4455498691690317961,INVESBOT,425405.92
1,2025-11-25,7190472578366945756,INVESBOT,7971987.66
2,2025-06-24,-1237314025466821076,INVESBOT,173043.65
3,2026-04-16,-375000770824817477,INVESBOT,27438543.23
4,2026-04-27,-2984135926251809763,INVESBOT,5121208.76



Resumen de estructuras cargadas:


,tabla,filas,columnas
0,clientes,860231,10
1,crean_aho_cte,1000000,4
2,crean_bolsillos,1000000,4
3,crean_fiducuenta,1000000,4
4,crean_inv_virtual_cdt,994177,4
5,estimador_ing,745792,3
6,invesbot,1000000,4


----------------------------------------------------

# Tener los .db en df para rapida visualización con DATAWRANGLER

In [4]:
clientes_df = dfs["clientes"]
crean_aho_cte_df = dfs["crean_aho_cte"]
crean_bolsillos_df = dfs["crean_bolsillos"]
crean_fiducuenta_df = dfs["crean_fiducuenta"]
crean_inv_virtual_cdt_df = dfs["crean_inv_virtual_cdt"]
estimador_ing_df = dfs["estimador_ing"]
invesbot_df = dfs["invesbot"]

---------------------------------------------------------------

# Diagnostico de integridad y duplicados

In [5]:
# 3. Auditoría de Duplicados en Llave Primaria (numero_id)
audit_list = []
base_ids = set(dfs["clientes"]["numero_id"].unique())

for name, df in dfs.items():
    total_filas = len(df)
    ids_unicos = df["numero_id"].nunique()
    duplicados_id = total_filas - ids_unicos
    nulos_id = df["numero_id"].isnull().sum()
    
    # Calculamos cuántos de sus IDs existen en la base maestra de clientes
    current_ids = set(df["numero_id"].dropna().unique())
    ids_en_maestra = len(current_ids.intersection(base_ids))
    overlap_pct = (ids_en_maestra / len(base_ids)) * 100 if len(base_ids) > 0 else 0
    
    audit_list.append({
        "Fuente": name,
        "Total Filas": total_filas,
        "IDs Únicos": ids_unicos,
        "Filas Duplicadas por ID": duplicados_id,
        "IDs Nulos": nulos_id,
        "Overlap con Clientes (%)": round(overlap_pct, 2)
    })

df_audit = pd.DataFrame(audit_list)
print("\n=== AUDITORÍA DE INTEGRIDAD Y COBERTURA DE CLIENTES ===")
display(df_audit)


=== AUDITORÍA DE INTEGRIDAD Y COBERTURA DE CLIENTES ===


,Fuente,Total Filas,IDs Únicos,Filas Duplicadas por ID,IDs Nulos,Overlap con Clientes (%)
0,clientes,860231,860223,8,0,100.00
1,crean_aho_cte,1000000,475719,524281,0,55.30
2,crean_bolsillos,1000000,260714,739286,0,30.31
3,crean_fiducuenta,1000000,181021,818979,0,21.04
4,crean_inv_virtual_cdt,994177,84104,910073,0,9.78
5,estimador_ing,745792,745792,0,0,86.70
6,invesbot,1000000,5214,994786,0,0.61


# Auditoría detallada de los duplicados en la tabla clientes

In [6]:
# 1. Identificar los IDs duplicados
dup_ids = clientes_df[clientes_df.duplicated(subset=["numero_id"], keep=False)]["numero_id"].unique()

print(f"=== DETECCIÓN DE CLIENTES DUPLICADOS ({len(dup_ids)} IDs afectados) ===")

# 2. Filtrar y mostrar las filas completas de los clientes duplicados
df_duplicados = clientes_df[clientes_df["numero_id"].isin(dup_ids)].sort_values(by="numero_id")

display(df_duplicados)

# 3. Diagnóstico de consistencia: ¿Son duplicados idénticos o tienen datos contradictorios?
es_duplicado_exacto = clientes_df.duplicated(keep=False).sum()

=== DETECCIÓN DE CLIENTES DUPLICADOS (8 IDs afectados) ===


,numero_id,grupo_edad,desc_genero,desc_segmento,desc_tipo_de_vivienda,ingresos_mensuales,total_egresos_mensuales,total_activos,total_pasivos,total_patrimonio
656476,-6607538840581733544,18-25,femenino,personal,NaN,2000000.00,200000.00,0.00,0.00,0.00
754528,-6607538840581733544,18-25,femenino,personal,NaN,2000000.00,200000.00,0.00,0.00,0.00
726190,-5638334313389883302,65+,femenino,personal,NaN,1880171.00,0.00,16066123.00,0.00,16066123.00
719409,-5638334313389883302,65+,femenino,personal,NaN,1880171.00,0.00,16066123.00,0.00,16066123.00
431390,-5555696474495328056,26-35,masculino,personal,NaN,1400000.00,1000000.00,4654000.00,0.00,0.00
335680,-5555696474495328056,26-35,masculino,personal,NaN,1400000.00,1000000.00,4654000.00,0.00,0.00
753262,-4381817681530763038,18-25,masculino,personal,NaN,1500000.00,400000.00,0.00,0.00,354000.00
645375,-4381817681530763038,18-25,masculino,personal,NaN,1500000.00,400000.00,0.00,0.00,354000.00
685153,-2314237362958234271,18-25,masculino,personal,NaN,2800000.00,100000.00,10000000.00,5800000.00,0.00
668044,-2314237362958234271,18-25,masculino,personal,NaN,2800000.00,100000.00,10000000.00,5800000.00,0.00


## Al ser ID duplicados de manera exacta, eliminamos para evitar repetidos

In [7]:
clientes_df = clientes_df.drop_duplicates(subset=['numero_id'], keep='first')

In [8]:
conteo_duplicado_id=clientes_df.duplicated(subset=['numero_id'], keep=False).sum()
print(conteo_duplicado_id)

0


------------------------------------------------------

# Conteo de valores nulos

In [9]:
# ==============================================================================
# AUDITORÍA DE VALORES NULOS POR TABLA Y COLUMNA
# ==============================================================================

nulos_list = []

for name, df in dfs.items():
    # Calculamos nulos por columna
    null_counts = df.isnull().sum()
    null_pcts = (df.isnull().mean()) * 100
    
    for col in df.columns:
        if null_counts[col] > 0:  # Omitimos columnas sin nulos para mantener foco
            nulos_list.append({
                "Tabla": name,
                "Columna": col,
                "Total Filas": len(df),
                "Cantidad Nulos": null_counts[col],
                "Porcentaje Nulos (%)": round(null_pcts[col], 2)
            })

df_nulos = pd.DataFrame(nulos_list)

print("=== INFORME DE VALORES NULOS DETECTADOS ===")
if not df_nulos.empty:
    display(df_nulos)
else:
    print(" No se detectaron valores nulos explícitos (NaN/None) en ninguna de las tablas.")

=== INFORME DE VALORES NULOS DETECTADOS ===


,Tabla,Columna,Total Filas,Cantidad Nulos,Porcentaje Nulos (%)
0,clientes,desc_genero,860231,93,0.01
1,clientes,desc_tipo_de_vivienda,860231,591699,68.78
2,clientes,ingresos_mensuales,860231,249,0.03
3,clientes,total_egresos_mensuales,860231,249,0.03
4,clientes,total_activos,860231,249,0.03
5,clientes,total_pasivos,860231,249,0.03
6,clientes,total_patrimonio,860231,260,0.03


## Significa que podrían ser clientes muy recientes, y por eso no cuentan con la informacion completa de la parte economica, no los voy a descartar porque podrían estar interesado en realizar inversiones así sean nuevos en la app, veremos si se pueden imputar desde la tabla "estimador_ing" para trabajar con mas informacion

In [12]:
# ==============================================================================
# VERIFICACIÓN DE COBERTURA: CLIENTES CON INGRESOS NULOS VS ESTIMADOR_ING
# ==============================================================================

# 1. Identificar los numero_id de los 249 clientes con ingresos_mensuales nulos
ids_ingresos_nulos = clientes_df[clientes_df["ingresos_mensuales"].isnull()]["numero_id"].unique()

# 2. Verificar cuántos de estos IDs existen en la tabla estimador_ing
ids_en_estimador = set(ids_ingresos_nulos).intersection(set(estimador_ing_df["numero_id"].unique()))

cant_nulos_totales = len(ids_ingresos_nulos)
cant_rescatables = len(ids_en_estimador)
cant_sin_estimador = cant_nulos_totales - cant_rescatables

print("=== COBERTURA DE COINCIDENCIAS PARA IMPUTACIÓN ===")
print(f"Total clientes con 'ingresos_mensuales' nulo: {cant_nulos_totales}")
print(f"Clientes encontrados en 'estimador_ing' (Imputables): {cant_rescatables} ({cant_rescatables/cant_nulos_totales*100:.2f}%)")
print(f"Clientes no encontrados en 'estimador_ing' (Imputables a 0): {cant_sin_estimador}")

# 3. Muestra de los registros que sí se pueden rescatar
if cant_rescatables > 0:
    print("\n--- MUESTRA DE VALORES A IMPUTAR DESDE ESTIMADOR_ING ---")
    muestra_estimaciones = estimador_ing_df[estimador_ing_df["numero_id"].isin(ids_en_estimador)].head(10)
    display(muestra_estimaciones[["numero_id", "producto", "estimador_ingreso"]])

=== COBERTURA DE COINCIDENCIAS PARA IMPUTACIÓN ===
Total clientes con 'ingresos_mensuales' nulo: 249
Clientes encontrados en 'estimador_ing' (Imputables): 176 (70.68%)
Clientes no encontrados en 'estimador_ing' (Imputables a 0): 73

--- MUESTRA DE VALORES A IMPUTAR DESDE ESTIMADOR_ING ---


,numero_id,producto,estimador_ingreso
2570,-534958678423269011,ESTIMADOR INGRESO,2464767.00
7426,5422248297671962158,ESTIMADOR INGRESO,1980123.98
11123,-304874487622358486,ESTIMADOR INGRESO,4620215.47
15146,5527100293230512876,ESTIMADOR INGRESO,1423500.00
16573,-6664945642711896222,ESTIMADOR INGRESO,2182828.00
19277,5383297359773286759,ESTIMADOR INGRESO,1542524.90
19645,-8631170758409885924,ESTIMADOR INGRESO,2273226.61
23374,268203900003158413,ESTIMADOR INGRESO,8798205.83
33642,-8802509204787021424,ESTIMADOR INGRESO,3315304.32
35093,-5681433734449377915,ESTIMADOR INGRESO,1566085.92


In [ ]:
# ==============================================================================
# IMPUTACIÓN EN DOS PASOS DE INGRESOS EN CLIENTES
# ==============================================================================

# 1. Consolidar estimador_ingreso a nivel numero_id (por si hay múltiples registros)
df_est_agg = estimador_ing_df.groupby("numero_id")["estimador_ingreso"].mean().reset_index()

# 2. Unir con la tabla clientes
clientes_df = clientes_df.merge(df_est_agg, on="numero_id", how="left")

# 3. Paso 1: Imputación con estimador_ingreso (rescata los 176 clientes)
clientes_df["ingresos_mensuales"] = clientes_df["ingresos_mensuales"].fillna(clientes_df["estimador_ingreso"])

# 4. Paso 2: Imputación a 0 para los no encontrados (los 73 clientes restantes)
clientes_df["ingresos_mensuales"] = clientes_df["ingresos_mensuales"].fillna(0)

# 6. Eliminar la columna auxiliar del merge
clientes_df = clientes_df.drop(columns=["estimador_ingreso"])

print("=== VERIFICACIÓN DE IMPUTACIÓN FINAL ===")
print(f"Nulos restantes en 'ingresos_mensuales': {clientes_df['ingresos_mensuales'].isnull().sum()}")
print(f"Clientes imputados a 0 (flag_sin_info_financiera = 1): {clientes_df['flag_sin_info_financiera'].sum()}")

=== VERIFICACIÓN DE IMPUTACIÓN FINAL ===
Nulos restantes en 'ingresos_mensuales': 0
Clientes imputados a 0 (flag_sin_info_financiera = 1): 9369


# *INSIGHT:* En la tabla clientes habían 249 valores nulos en el campo "ingresos_mensuales", se imputaron de la tabla "estimador_ing" los posibles valores mediante un cruce por numero_id. Con esto se recuperaron 176 registros (70.68%), mientras que los 73 restantes (29.32%) sin historial se asignaron en $0$

--------------------------------------------------------

# Analisis de clientes con patrimonio negativo

## patrimonio = activos - pasivos

In [14]:
# 1. Conteo y porcentaje de patrimonio negativo (< 0)
patrimonio_neg = clientes_df[clientes_df["total_patrimonio"] < 0]
cant_neg = len(patrimonio_neg)
total_clientes = len(clientes_df)
pct_neg = (cant_neg / total_clientes) * 100

print("=== INFORME DE PATRIMONIO NEGATIVO (< 0) ===")
print(f"Total de registros evaluados: {total_clientes:,}")
print(f"Cantidad de clientes con patrimonio < 0: {cant_neg:,}")
print(f"Porcentaje sobre el total: {pct_neg:.2f}%")

=== INFORME DE PATRIMONIO NEGATIVO (< 0) ===
Total de registros evaluados: 860,223
Cantidad de clientes con patrimonio < 0: 6,173
Porcentaje sobre el total: 0.72%


# Calculo manual para valida los patrimonios negativos

In [15]:
# 1. Cálculo de la variable derivada del patrimonio contable
patrimonio_calc = clientes_df["total_activos"] - clientes_df["total_pasivos"]

# 2. Conteo de registros donde el patrimonio calculado es menor a 0
cant_patrimonio_neg = (patrimonio_calc < 0).sum()
total_registros = len(clientes_df)
pct_patrimonio_neg = (cant_patrimonio_neg / total_registros) * 100

print(f"Total de clientes analizados: {total_registros:,}")
print(f"Clientes con patrimonio negativo (Activos - Pasivos < 0): {cant_patrimonio_neg:,}")
print(f"Porcentaje sobre el total: {pct_patrimonio_neg:.2f}%")

Total de clientes analizados: 860,223
Clientes con patrimonio negativo (Activos - Pasivos < 0): 11,009
Porcentaje sobre el total: 1.28%


# *INSIGHT:* Ese desface puede ser algo normal, puede suceder porque puede que se actualice mas rapido el campo de pasivos que el campo del patrimonio, o puede que hayan algunas reglas internas de ajustes de riesgo o patrimonio intangible

# Se van a trabajar igualmente con los datos que tienen patrimonio en 0, ya que eso no significa que no tenga, es muy probable que no los tenga registrado a su nombre

-----------------------------------------------------------

# Analisis de las cuentas de ahorro y cuentas corrientes con saldos negativos

In [17]:
ahorro_negativo = crean_aho_cte_df[
    (crean_aho_cte_df["producto"].str.upper() == "CUENTA DE AHORRO") & 
    (crean_aho_cte_df["saldo"] < 0)
]

cant_ahorro_neg = len(ahorro_negativo)
clientes_unicos_ahorro_neg = ahorro_negativo["numero_id"].nunique()
total_registros_ahorro = len(crean_aho_cte_df[crean_aho_cte_df["producto"].str.upper() == "CUENTA DE AHORRO"])

print("=== INFORME DE CUENTAS DE AHORRO EN NEGATIVO ===")
print(f"Total registros de Cuenta de Ahorro: {total_registros_ahorro:,}")
print(f"Registros en negativo (< 0): {cant_ahorro_neg:,} ({cant_ahorro_neg/total_registros_ahorro*100:.4f}%)")
print(f"Clientes únicos afectados: {clientes_unicos_ahorro_neg:,}")

# 2. Inspeccionar estadísticas del saldo negativo
if cant_ahorro_neg > 0:
       
    print("\n--- MUESTRA DE REGISTROS AFECTADOS ---")
    display(ahorro_negativo[["fecha", "numero_id", "producto", "saldo"]].head(10))

=== INFORME DE CUENTAS DE AHORRO EN NEGATIVO ===
Total registros de Cuenta de Ahorro: 987,914
Registros en negativo (< 0): 3 (0.0003%)
Clientes únicos afectados: 3

--- MUESTRA DE REGISTROS AFECTADOS ---


,fecha,numero_id,producto,saldo
613484,2025-12-01,-9036810433992029731,CUENTA DE AHORRO,-0.57
696121,2026-03-01,-1849271277322924224,CUENTA DE AHORRO,-2911.62
932088,2025-08-01,4213766447344583291,CUENTA DE AHORRO,-0.06


In [18]:
# 1. Filtrar únicamente los registros de Cuenta Corriente
corriente_df = crean_aho_cte_df[crean_aho_cte_df["producto"].str.upper() == "CUENTA DE CORRIENTE"].copy()

total_corriente = len(corriente_df)
clientes_corriente = corriente_df["numero_id"].nunique()

# 2. Partición por signo de saldo
positos_corriente = corriente_df[corriente_df["saldo"] >= 0]
negativos_corriente = corriente_df[corriente_df["saldo"] < 0]

cant_pos = len(positos_corriente)
cant_neg = len(negativos_corriente)

pct_pos = (cant_pos / total_corriente) * 100 if total_corriente > 0 else 0
pct_neg = (cant_neg / total_corriente) * 100 if total_corriente > 0 else 0

print("=== INFORME DE CUENTAS CORRIENTES ===")
print(f"Total registros de Cuenta Corriente: {total_corriente:,}")
print(f"Clientes únicos con Cuenta Corriente: {clientes_corriente:,}")
print("-" * 50)
print(f"Saldos Positivos o Cero (>= 0): {cant_pos:,} ({pct_pos:.2f}%)")
print(f"Saldos Negativos / Sobregiros (< 0): {cant_neg:,} ({pct_neg:.2f}%)")

# 3. Estadísticas descriptivas de los sobregiros (< 0)
if cant_neg > 0:
    print("\n--- MAGNITUD DE LOS SOBREGIROS EN CUENTA CORRIENTE (COP) ---")
    print(f"Sobregiro Máximo (Mayor cupo usado): ${negativos_corriente['saldo'].min():,.2f}")
    print(f"Sobregiro Promedio: ${negativos_corriente['saldo'].mean():,.2f}")
    print(f"Sobregiro Mediano: ${negativos_corriente['saldo'].median():,.2f}")
    
    print("\n--- MUESTRA DE REGISTROS CON SOBREGIRO ---")
    display(negativos_corriente[["fecha", "numero_id", "producto", "saldo"]].head(10))

=== INFORME DE CUENTAS CORRIENTES ===
Total registros de Cuenta Corriente: 12,086
Clientes únicos con Cuenta Corriente: 5,645
--------------------------------------------------
Saldos Positivos o Cero (>= 0): 10,619 (87.86%)
Saldos Negativos / Sobregiros (< 0): 1,467 (12.14%)

--- MAGNITUD DE LOS SOBREGIROS EN CUENTA CORRIENTE (COP) ---
Sobregiro Máximo (Mayor cupo usado): $-20,281,714.09
Sobregiro Promedio: $-1,209,538.55
Sobregiro Mediano: $-385,847.77

--- MUESTRA DE REGISTROS CON SOBREGIRO ---


,fecha,numero_id,producto,saldo
458,2026-06-01,-3049228187875985986,CUENTA DE CORRIENTE,-395712.45
594,2026-06-01,-3089205321861291540,CUENTA DE CORRIENTE,-359038.57
859,2025-08-01,-4659440375858593297,CUENTA DE CORRIENTE,-62942.92
908,2026-06-01,-6550551175028050669,CUENTA DE CORRIENTE,-1940835.31
966,2026-02-01,5915716075441812949,CUENTA DE CORRIENTE,-126015.21
1018,2026-06-01,6819867896480291287,CUENTA DE CORRIENTE,-1007360.96
1309,2025-09-01,-2273759262996810650,CUENTA DE CORRIENTE,-14338258.31
1429,2026-06-01,3425534458937859670,CUENTA DE CORRIENTE,-749546.43
1937,2025-11-01,-3095866817426729569,CUENTA DE CORRIENTE,-1752910.96
3822,2026-04-01,-2696208820413282920,CUENTA DE CORRIENTE,-3098847.62


# *INSIGHT:* Al tener saldos negativos, automaticamente se deben de descartar esas cuentas, porque para poder realizar inversiones la app primero validará el dinero real disponible en la cuenta

## Fuente: https://www.bancolombia.com/centro-de-ayuda/preguntas-frecuentes/solicitar-inversion-virtual-app-personas

--------------------------------------------

# Analisis para las inversiones virtuales y los CDT que tienen saldo = 0

In [19]:
# 1. Filtrar registros donde el saldo es estrictamente igual a 0
saldos_cero = crean_inv_virtual_cdt_df[crean_inv_virtual_cdt_df["saldo"] == 0]

total_registros = len(crean_inv_virtual_cdt_df)
cant_cero = len(saldos_cero)
pct_cero = (cant_cero / total_registros) * 100

print("=== INFORME DE SALDOS EN CERO (0.0) EN INVERSIÓN VIRTUAL / CDT ===")
print(f"Total registros analizados: {total_registros:,}")
print(f"Registros con saldo = 0: {cant_cero:,} ({pct_cero:.2f}%)")
print(f"Clientes únicos afectados: {saldos_cero['numero_id'].nunique():,}")

# 2. Desglose de saldos en cero por tipo de producto (CDT vs Inversión Virtual)
if cant_cero > 0:
    print("\n--- DISTRIBUCIÓN POR PRODUCTO ---")
    conteo_producto = saldos_cero["producto"].value_counts().reset_index()
    conteo_producto.columns = ["Producto", "Cantidad en Cero"]
    conteo_producto["Porcentaje (%)"] = (conteo_producto["Cantidad en Cero"] / cant_cero) * 100
    display(conteo_producto)
    
    print("\n--- MUESTRA DE REGISTROS CON SALDO = 0 ---")
    display(saldos_cero[["fecha", "numero_id", "producto", "saldo"]].head(10))

=== INFORME DE SALDOS EN CERO (0.0) EN INVERSIÓN VIRTUAL / CDT ===
Total registros analizados: 994,177
Registros con saldo = 0: 4,017 (0.40%)
Clientes únicos afectados: 3,102

--- DISTRIBUCIÓN POR PRODUCTO ---


,Producto,Cantidad en Cero,Porcentaje (%)
0,INVERSIóN VIRTUAL,4017,100.00



--- MUESTRA DE REGISTROS CON SALDO = 0 ---


,fecha,numero_id,producto,saldo
180,2025-09-01,6071014405761666485,INVERSIóN VIRTUAL,0.00
296,2025-08-01,-2131313075955300613,INVERSIóN VIRTUAL,0.00
338,2026-02-01,3393495603739633426,INVERSIóN VIRTUAL,0.00
553,2026-02-01,-8323120750407885131,INVERSIóN VIRTUAL,0.00
814,2026-03-01,4372706901624664198,INVERSIóN VIRTUAL,0.00
1157,2026-02-01,1517742472873310487,INVERSIóN VIRTUAL,0.00
1167,2026-05-01,7769590751761434813,INVERSIóN VIRTUAL,0.00
1215,2025-08-01,616526556411235208,INVERSIóN VIRTUAL,0.00
1404,2026-05-01,4009695997575164509,INVERSIóN VIRTUAL,0.00
1413,2025-12-01,-4431469699822784481,INVERSIóN VIRTUAL,0.00


# INSIGHT: Así se encuentren registros con saldo = 0 en el producto de Inversion Virtual significa que en algún momento se interesaron en realizar una inversión, por lo que pueden ser potencial clientes para la nueva app de inversiones

-------------------------------------------------

# *INSIGHT:* Las tablas (crean_aho_cte, crean_fiducuenta, crean_bolsillos) todos tienen fecha del primer dia de cada mes, puede ser por la fecha de ingestion o de actualizacion de las tablas

-------------------------------------------

In [20]:
# 1. Filtrar registros donde el saldo es estrictamente igual a 0
saldos_cero = crean_bolsillos_df[crean_bolsillos_df["saldo"] == 0]

total_registros = len(crean_bolsillos_df)
cant_cero = len(saldos_cero)
pct_cero = (cant_cero / total_registros) * 100

print("=== INFORME DE SALDOS EN CERO (0.0) EN INVERSIÓN VIRTUAL / CDT ===")
print(f"Total registros analizados: {total_registros:,}")
print(f"Registros con saldo = 0: {cant_cero:,} ({pct_cero:.2f}%)")
print(f"Clientes únicos afectados: {saldos_cero['numero_id'].nunique():,}")

# 2. Desglose de saldos en cero por tipo de producto (CDT vs Inversión Virtual)
if cant_cero > 0:
    print("\n--- DISTRIBUCIÓN POR PRODUCTO ---")
    conteo_producto = saldos_cero["producto"].value_counts().reset_index()
    conteo_producto.columns = ["Producto", "Cantidad en Cero"]
    conteo_producto["Porcentaje (%)"] = (conteo_producto["Cantidad en Cero"] / cant_cero) * 100
    display(conteo_producto)
    
    print("\n--- MUESTRA DE REGISTROS CON SALDO = 0 ---")
    display(saldos_cero[["fecha", "numero_id", "producto", "saldo"]].head(10))

=== INFORME DE SALDOS EN CERO (0.0) EN INVERSIÓN VIRTUAL / CDT ===
Total registros analizados: 1,000,000
Registros con saldo = 0: 467,155 (46.72%)
Clientes únicos afectados: 160,762

--- DISTRIBUCIÓN POR PRODUCTO ---


,Producto,Cantidad en Cero,Porcentaje (%)
0,BOLSILLOS,467155,100.00



--- MUESTRA DE REGISTROS CON SALDO = 0 ---


,fecha,numero_id,producto,saldo
0,2026-05-01,1013449549804401435,BOLSILLOS,0.00
4,2026-01-01,1547171292443722937,BOLSILLOS,0.00
9,2025-10-01,-7673223733260382286,BOLSILLOS,0.00
10,2026-06-01,-7037377401071587964,BOLSILLOS,0.00
11,2025-12-01,3676312238173092759,BOLSILLOS,0.00
15,2025-11-01,3887635984691863752,BOLSILLOS,0.00
16,2026-06-01,5330806333830989882,BOLSILLOS,0.00
20,2026-01-01,-2425833549847612833,BOLSILLOS,0.00
21,2026-02-01,68005637688305040,BOLSILLOS,0.00
22,2025-12-01,-1197931878267528635,BOLSILLOS,0.00


# *INSIGHT:* Mismo caso que en inversion virtual_CDT, un bolsillo en 0 no significa que no tenga dinero, significa que en algun momento se intereso en el producto y realizo la apertura. Tambien lo catalogo como cliente potencial y por eso lo tomaré en cuenta para el modelo